In [49]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/test.csv')
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,id,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,...,pc,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi
0,1,1043,1,1.8,1,14,0,5,0.1,193,...,16,226,1412,3476,12,7,2,0,1,0
1,2,841,1,0.5,1,4,1,61,0.8,191,...,12,746,857,3895,6,0,7,1,0,0
2,3,1807,1,2.8,0,1,0,27,0.9,186,...,4,1270,1366,2396,17,10,10,0,1,1
3,4,1546,0,0.5,1,18,1,25,0.5,96,...,20,295,1752,3893,10,0,7,1,1,0
4,5,1434,0,1.4,0,11,1,49,0.5,108,...,18,749,810,1773,15,8,7,1,0,1


## Data preprocessing




**Reasoning**:
Check for missing values in the dataframe.



In [50]:
df.isnull().sum()

,0
id,0
battery_power,0
blue,0
clock_speed,0
dual_sim,0
fc,0
four_g,0
int_memory,0
m_dep,0
mobile_wt,0


**Reasoning**:
Since there are no missing values, the next step is to identify and encode categorical features and then scale the numerical features.



In [51]:
# Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=['object']).columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Separate features (X) and target (y) before scaling
# Using 'four_g' as the target variable
y = df['four_g']
X = df.drop('four_g', axis=1)


# Exclude the target variable 'four_g' from numerical columns for scaling (already done by dropping it for X)
# numerical_cols = numerical_cols.drop('four_g')


# No categorical columns to encode in this dataset

# Scale numerical features in X
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X[numerical_cols.drop('four_g', errors='ignore')] = scaler.fit_transform(X[numerical_cols.drop('four_g', errors='ignore')])

# The original df is not used after splitting, so no need to display df.head()
# display(df.head())
display(X.head())
display(y.head())

,id,battery_power,blue,clock_speed,dual_sim,fc,int_memory,m_dep,mobile_wt,n_cores,pc,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi
0,-1.730320,-0.475451,0.968496,0.312601,0.966559,2.108676,-1.581269,-1.487247,1.535535,-0.580671,0.976026,-0.926990,0.391912,1.229373,0.001158,0.397363,-1.653355,-1.760216,1.0,-1.014099
1,-1.726856,-0.942782,0.968496,-1.255832,0.966559,-0.132927,1.509303,1.006341,1.478120,0.293833,0.319433,0.274729,-0.871028,1.614643,-1.388231,-1.254383,-0.743418,0.568112,-1.0,-1.014099
2,-1.723391,1.292077,0.968496,1.519087,-1.034598,-0.805408,-0.367116,1.362567,1.334582,-0.580671,-0.993754,1.485693,0.287236,0.236313,1.158982,1.105254,-0.197456,-1.760216,1.0,0.986097
3,-1.719927,0.688249,-1.032529,-1.255832,0.966559,3.005317,-0.477493,-0.062340,-1.249091,1.605590,1.632619,-0.767532,1.165604,1.612804,-0.461972,-1.254383,-0.743418,0.568112,1.0,-1.014099
4,-1.716463,0.429135,-1.032529,-0.169994,-1.034598,1.436195,0.847037,-0.062340,-0.904602,0.731085,1.304323,0.281662,-0.977979,-0.336535,0.695852,0.633326,-0.743418,0.568112,-1.0,0.986097


,four_g
0,0
1,1
2,0
3,1
4,1


**Reasoning**:
Separate features and target, split data, instantiate and train multiple classifiers.



In [52]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, BaggingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import SGDClassifier
import lightgbm as lgb
import xgboost as xgb

# Separate features (X) and target (y) BEFORE scaling
# Using 'four_g' as the target variable
y = df['four_g']
X = df.drop('four_g', axis=1)

# Identify numerical columns in X for scaling
numerical_cols_X = X.select_dtypes(include=['int64', 'float64']).columns

# Scale numerical features in X
scaler = StandardScaler()
X[numerical_cols_X] = scaler.fit_transform(X[numerical_cols_X])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Instantiate and train each classifier
classifiers = {
    "Logistic Regression": LogisticRegression(),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Support Vector Machine": SVC(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gaussian Naive Bayes": GaussianNB(),
    "SGD Classifier": SGDClassifier(),
    "AdaBoost Classifier": AdaBoostClassifier(),
    "Bagging Classifier": BaggingClassifier(),
    "LightGBM": lgb.LGBMClassifier(),
    "XGBoost": xgb.XGBClassifier()
}

trained_models = {}
for name, clf in classifiers.items():
    print(f"Training {name}...")
    # Add print statements to check y_train before fitting
    print(f"  y_train dtype: {y_train.dtype}")
    print(f"  y_train unique values: {y_train.unique()}")
    clf.fit(X_train, y_train)
    trained_models[name] = clf
    print(f"{name} trained.")

print("\nAll models trained successfully.")

Training Logistic Regression...
  y_train dtype: int64
  y_train unique values: [1 0]
Logistic Regression trained.
Training K-Nearest Neighbors...
  y_train dtype: int64
  y_train unique values: [1 0]
K-Nearest Neighbors trained.
Training Support Vector Machine...
  y_train dtype: int64
  y_train unique values: [1 0]
Support Vector Machine trained.
Training Decision Tree...
  y_train dtype: int64
  y_train unique values: [1 0]
Decision Tree trained.
Training Random Forest...
  y_train dtype: int64
  y_train unique values: [1 0]
Random Forest trained.
Training Gaussian Naive Bayes...
  y_train dtype: int64
  y_train unique values: [1 0]
Gaussian Naive Bayes trained.
Training SGD Classifier...
  y_train dtype: int64
  y_train unique values: [1 0]
SGD Classifier trained.
Training AdaBoost Classifier...
  y_train dtype: int64
  y_train unique values: [1 0]
AdaBoost Classifier trained.
Training Bagging Classifier...
  y_train dtype: int64
  y_train unique values: [1 0]
Bagging Classifier tr

In [53]:
# Select a trained model (e.g., K-Nearest Neighbors)
model_to_use = trained_models["K-Nearest Neighbors"]

# Take a sample from the test set
# You can choose a different index to predict for a different test point
sample_to_predict_df = X_test.iloc[[0]] # Keep as DataFrame to retain feature names

# Make a prediction
prediction = model_to_use.predict(sample_to_predict_df)

print(f"The predicted 'four_g' value for the sample is: {prediction[0]}")

The predicted 'four_g' value for the sample is: 1


In [54]:
from sklearn.metrics import accuracy_score

# Evaluate each trained model
print("Model Accuracy on Test Set:")
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"- {name}: {accuracy:.4f}")

Model Accuracy on Test Set:
- Logistic Regression: 0.6950
- K-Nearest Neighbors: 0.6250
- Support Vector Machine: 0.6950
- Decision Tree: 0.6150
- Random Forest: 0.6950
- Gaussian Naive Bayes: 0.6900
- SGD Classifier: 0.6450
- AdaBoost Classifier: 0.6900
- Bagging Classifier: 0.6500
- LightGBM: 0.6550
- XGBoost: 0.6300
